# Templat Notebook Projek Akhir — Mini-Project Kecerdasan Buatan
**PIF403 Pengantar Kecerdasan Buatan · Prodi Informatika Universitas Adzkia · Pertemuan 12–16**

Notebook ini adalah **kerangka kerja** mini-project kelompok. Susunannya mengikuti rubrik A6 (implementasi), dan isinya menjadi bahan laporan akhir (A7) serta demonstrasi (A8).

| Bagian | Isi | Kriteria rubrik A6 |
|---|---|---|
| 1 | Identitas dan deskripsi masalah | — |
| 2 | Formulasi (PEAS, representasi) | Formulasi 15% |
| 3 | Metode dan implementasi | Metode 25%, Implementasi 25% |
| 4 | Pengujian dan evaluasi | Pengujian 20% |
| 5 | Reproducibility dan log kerja | Reproducibility 5% |
| 6 | Etika dan keterbatasan (system card) | Etika 10% |

**Cara pakai:** setiap sel bertanda `CONTOH` berisi contoh kecil yang berjalan (asisten diagnosis laptop). **Ganti seluruh contoh dengan sistem kelompok Anda.** Fungsi pada bagian "Perkakas" boleh dipakai apa adanya.

## 1. Identitas dan deskripsi masalah

| | |
|---|---|
| **Judul projek** | ... |
| **Kelompok** | ... |
| **Anggota (NIM, nama, peran)** | 1. ... 2. ... 3. ... |
| **Pengguna sasaran** | siapa yang akan memakai sistem ini? |
| **Masalah yang diselesaikan** | 2–3 kalimat |
| **Mengapa perlu AI** | mengapa tidak cukup dengan program biasa atau tabel? |
| **Metode AI yang dipakai** | minimal dua dari: pencarian, CSP, aturan/logika, probabilistik |

In [ ]:
# Perkakas 1: pencatat lingkungan dan pengendali keacakan (reproducibility)
import sys, platform, random, time, json, datetime

BENIH = 403                       # satu angka benih untuk seluruh projek
random.seed(BENIH)

def info_lingkungan():
    import importlib
    info = {"python": sys.version.split()[0], "platform": platform.platform(), "benih": BENIH,
            "waktu_jalan": datetime.datetime.now().isoformat(timespec="seconds")}
    for pustaka in ("numpy", "pandas", "networkx", "matplotlib"):
        try: info[pustaka] = importlib.import_module(pustaka).__version__
        except ImportError: info[pustaka] = "tidak terpasang"
    return info

LINGKUNGAN = info_lingkungan()
print(json.dumps(LINGKUNGAN, indent=2))

In [ ]:
# Perkakas 2: log keputusan agen. Setiap keputusan penting dicatat agar bisa ditelusuri ulang.
LOG = []

def catat(tahap, pesan, **data):
    """Mencatat satu peristiwa. tahap: 'persepsi' | 'penalaran' | 'tindakan' | 'galat'."""
    LOG.append({"no": len(LOG) + 1, "tahap": tahap, "pesan": pesan, **data})

def tampilkan_log(n=20):
    for baris in LOG[-n:]:
        ekstra = {k: v for k, v in baris.items() if k not in ("no", "tahap", "pesan")}
        print(f"[{baris['no']:03d}] {baris['tahap']:<10} {baris['pesan']} {ekstra if ekstra else ''}")

def simpan_log(nama_berkas="log_agen.json"):
    with open(nama_berkas, "w", encoding="utf-8") as f:
        json.dump({"lingkungan": LINGKUNGAN, "log": LOG}, f, ensure_ascii=False, indent=2)
    print("Log disimpan ke", nama_berkas)

## 2. Formulasi

### 2.1 PEAS
| Komponen | Uraian |
|---|---|
| **Performance** (ukuran kinerja) | ... |
| **Environment** (lingkungan) | ... |
| **Actuators** (keluaran/tindakan) | ... |
| **Sensors** (masukan) | ... |

Sifat lingkungan: *teramati penuh/sebagian, deterministik/stokastik, episodik/sekuensial, statis/dinamis, diskret/kontinu, agen tunggal/jamak*. Jelaskan pilihan Anda.

### 2.2 Representasi
Tuliskan bentuk status, basis pengetahuan, atau variabel yang dipakai, beserta **asumsi** dan **sumber pengetahuan/data**.

| Unsur | Representasi | Sumber | Asumsi |
|---|---|---|---|
| ... | ... | ... | ... |

## 3. Metode dan implementasi

Susun sistem mengikuti arsitektur agen: **persepsi → pengetahuan → penalaran → tindakan**. Setiap tahap adalah fungsi tersendiri agar mudah diuji.

```
masukan pengguna ──► persepsi() ──► penalaran() ──► tindakan() ──► keluaran + penjelasan
                                        ▲
                                 basis pengetahuan
```

In [ ]:
# CONTOH — ganti dengan basis pengetahuan kelompok Anda
ATURAN = [
    dict(id="R1", jika=["tidak_menyala", "lampu_daya_mati"], maka="masalah_daya", alasan="Arus tidak sampai ke papan induk."),
    dict(id="R2", jika=["masalah_daya", "lampu_charger_mati"], maka="adaptor_rusak", alasan="Charger tidak mengeluarkan daya."),
    dict(id="R3", jika=["mati_mendadak", "bodi_panas"], maka="overheat", alasan="Pengaman suhu mematikan laptop."),
]
GEJALA_DIKENAL = sorted({p for a in ATURAN for p in a["jika"]} - {a["maka"] for a in ATURAN})
DIAGNOSIS = {"adaptor_rusak", "overheat"}
# Keyakinan awal tiap diagnosis (prior) — CONTOH angka perkiraan, tulis sumbernya pada laporan
PRIOR = {"adaptor_rusak": 0.15, "overheat": 0.25}

In [ ]:
# CONTOH — tiga tahap agen
def persepsi(masukan):
    """Membersihkan masukan dan memisahkan gejala yang dikenal dari yang tidak dikenal."""
    gejala = [g.strip().lower().replace(" ", "_") for g in masukan]
    dikenal = [g for g in gejala if g in GEJALA_DIKENAL]
    asing = [g for g in gejala if g not in GEJALA_DIKENAL]
    catat("persepsi", "masukan diproses", dikenal=dikenal, asing=asing)
    return dikenal, asing

def penalaran(fakta):
    """Forward chaining dengan pencatatan asal setiap kesimpulan."""
    memori, asal = list(fakta), {}
    berubah = True
    while berubah:
        berubah = False
        for a in ATURAN:
            if a["maka"] not in memori and all(p in memori for p in a["jika"]):
                memori.append(a["maka"]); asal[a["maka"]] = a; berubah = True
                catat("penalaran", f"aturan {a['id']} dijalankan", kesimpulan=a["maka"])
    return memori, asal

def tindakan(memori, asal, asing):
    """Menyusun keluaran: diagnosis, penjelasan, dan pengakuan batas pengetahuan."""
    hasil = [f for f in memori if f in DIAGNOSIS]
    if not hasil:
        catat("tindakan", "tidak ada diagnosis")
        return {"diagnosis": None, "pesan": "Belum dapat disimpulkan. Silakan konsultasi ke teknisi.", "diabaikan": asing}
    penjelasan = []
    for d in hasil:
        a = asal[d]; penjelasan.append(f"{d}: {a['alasan']} (aturan {a['id']}, dari {a['jika']})")
    catat("tindakan", "diagnosis diberikan", diagnosis=hasil)
    return {"diagnosis": hasil, "penjelasan": penjelasan, "diabaikan": asing}

def agen(masukan):
    """Satu siklus agen. Galat tidak boleh membuat sistem berhenti tanpa pesan."""
    try:
        dikenal, asing = persepsi(masukan)
        memori, asal = penalaran(dikenal)
        return tindakan(memori, asal, asing)
    except Exception as galat:
        catat("galat", str(galat), masukan=masukan)
        return {"diagnosis": None, "pesan": f"Terjadi galat: {galat}"}

print(agen(["Tidak menyala", "lampu daya mati", "lampu charger mati", "layar retak"]))
print(); tampilkan_log()

## 4. Pengujian dan evaluasi

Rencana evaluasi (A5) menentukan **apa yang diukur** sebelum sistem diuji. Minimal mencakup:

| Aspek | Pertanyaan | Contoh ukuran |
|---|---|---|
| Correctness | Apakah keluarannya benar? | persentase kasus uji yang lulus |
| Completeness | Apakah sistem selalu memberi jawaban bila jawaban ada? | jumlah kasus "tidak terjawab" |
| Optimality | Apakah solusinya yang terbaik? | selisih biaya terhadap solusi optimal |
| Efisiensi | Seberapa mahal? | waktu, jumlah node/aturan diproses |
| Robustness | Bagaimana bila masukan aneh, kosong, atau saling bertentangan? | jumlah galat tak tertangani |
| Baseline | Lebih baik dari cara sederhana? | perbandingan dengan baseline |

**Kasus uji minimal 15**, terdiri atas kasus normal, kasus batas, dan kasus tidak sah.

In [ ]:
# Perkakas 3: pelari kasus uji dengan pembanding baseline
import pandas as pd

def jalankan_uji(kasus_uji, sistem, baseline=None, ambil=lambda keluaran: keluaran):
    """kasus_uji: list of dict(nama, jenis, masukan, harapan). ambil: fungsi yang mengambil jawaban dari keluaran sistem."""
    baris = []
    for k in kasus_uji:
        t = time.perf_counter()
        try:
            jawab = ambil(sistem(k["masukan"])); galat = ""
        except Exception as e:
            jawab, galat = None, str(e)
        durasi = (time.perf_counter() - t) * 1000
        rekam = {"kasus": k["nama"], "jenis": k["jenis"], "harapan": k["harapan"], "sistem": jawab,
                 "lulus": jawab == k["harapan"], "ms": round(durasi, 2), "galat": galat}
        if baseline:
            rekam["baseline"] = baseline(k["masukan"]); rekam["baseline lulus"] = rekam["baseline"] == k["harapan"]
        baris.append(rekam)
    return pd.DataFrame(baris)

In [ ]:
# CONTOH — ganti dengan kasus uji kelompok Anda (minimal 15)
KASUS_UJI = [
    dict(nama="U01 adaptor", jenis="normal", masukan=["tidak_menyala", "lampu_daya_mati", "lampu_charger_mati"], harapan=["adaptor_rusak"]),
    dict(nama="U02 overheat", jenis="normal", masukan=["mati_mendadak", "bodi_panas"], harapan=["overheat"]),
    dict(nama="U03 dua gangguan", jenis="normal", masukan=["tidak_menyala", "lampu_daya_mati", "lampu_charger_mati", "mati_mendadak", "bodi_panas"],
         harapan=["adaptor_rusak", "overheat"]),
    dict(nama="U04 gejala kurang", jenis="batas", masukan=["tidak_menyala"], harapan=None),
    dict(nama="U05 masukan kosong", jenis="batas", masukan=[], harapan=None),
    dict(nama="U06 huruf besar dan spasi", jenis="batas", masukan=["Mati Mendadak", "BODI PANAS"], harapan=["overheat"]),
    dict(nama="U07 gejala asing", jenis="tidak sah", masukan=["keyboard_basah"], harapan=None),
    dict(nama="U08 kasus di luar aturan", jenis="normal", masukan=["tidak_menyala", "lampu_daya_mati", "lampu_charger_menyala"], harapan=["baterai_rusak"]),
]

def baseline_terbanyak(masukan):
    """Baseline sederhana: selalu menjawab diagnosis yang paling sering terjadi."""
    return [max(PRIOR, key=PRIOR.get)]

hasil_uji = jalankan_uji(KASUS_UJI, agen, baseline_terbanyak, ambil=lambda k: k["diagnosis"])
print(hasil_uji[["kasus", "jenis", "harapan", "sistem", "lulus", "baseline lulus"]].to_string(index=False))
print(f"\nSistem  : {hasil_uji['lulus'].mean():.0%} lulus ({hasil_uji['lulus'].sum()} dari {len(hasil_uji)})")
print(f"Baseline: {hasil_uji['baseline lulus'].mean():.0%} lulus")

### 4.1 Analisis galat
Kasus yang gagal adalah temuan paling berharga. Untuk **setiap** kasus gagal, isi tabel berikut (contoh diisi untuk U08).

| Kasus | Gejala kegagalan | Akar masalah | Rencana perbaikan | Status |
|---|---|---|---|---|
| U08 | sistem menjawab "belum dapat disimpulkan", seharusnya `baterai_rusak` | belum ada aturan untuk `lampu_charger_menyala` | tambah aturan R4 dan gejalanya | terbuka |
| ... | ... | ... | ... | ... |

In [ ]:
gagal = hasil_uji[~hasil_uji["lulus"]]
print("Kasus gagal:"); print(gagal[["kasus", "harapan", "sistem", "galat"]].to_string(index=False))
print("\nKelulusan per jenis kasus:"); print(hasil_uji.groupby("jenis")["lulus"].agg(["sum", "count"]).to_string())

### 4.2 Batas generalisasi
Jelaskan: pada kondisi apa sistem ini **tidak boleh** dipercaya? Kasus seperti apa yang belum diuji?

*Jawaban:*

## 5. Reproducibility dan log kerja

Daftar periksa sebelum mengumpulkan:
- [ ] *Runtime → Restart and run all* berjalan tanpa galat
- [ ] Benih acak ditetapkan satu kali di awal
- [ ] Versi pustaka tercatat (lihat `LINGKUNGAN`)
- [ ] Semua berkas data disertakan dan sumbernya ditulis
- [ ] Log agen tersimpan (`simpan_log()`)

### Log kerja kelompok
| Tanggal | Kegiatan | Dikerjakan oleh | Hasil / keputusan | Bukti (sel, berkas, commit) |
|---|---|---|---|---|
| ... | ... | ... | ... | ... |

In [ ]:
simpan_log()

## 6. Etika dan keterbatasan: system card

| Butir | Uraian |
|---|---|
| **Tujuan penggunaan** | untuk apa sistem ini dibuat |
| **Di luar cakupan** | penggunaan yang tidak didukung atau berbahaya |
| **Pengguna dan pihak terdampak** | siapa yang memakai, siapa yang terkena akibat keputusan |
| **Sumber pengetahuan/data** | asal, tanggal, cara memperoleh, izin |
| **Metode** | ringkasan metode AI yang dipakai |
| **Hasil evaluasi** | angka utama, baseline, jumlah kasus uji |
| **Keterbatasan yang diketahui** | kasus gagal, asumsi, cakupan pengetahuan |
| **Risiko bias dan ketidakadilan** | kelompok mana yang mungkin dirugikan, mengapa |
| **Privasi dan keamanan** | data pribadi yang disentuh, cara melindunginya |
| **Transparansi** | bagaimana sistem menjelaskan keputusannya |
| **Pengawasan manusia** | keputusan apa yang tetap harus diambil manusia |
| **Kontak dan versi** | penanggung jawab, tanggal, nomor versi |

## 7. Kontribusi individu

| Anggota | Bagian yang dikerjakan | Perkiraan porsi (%) | Bukti |
|---|---|---|---|
| ... | ... | ... | ... |

Pernyataan keaslian: *Kami menyatakan bahwa projek ini adalah hasil kerja kelompok kami. Bantuan dari pihak lain, sumber kode, dan penggunaan alat AI generatif kami sebutkan secara jujur pada bagian berikut.*

**Bantuan dan sumber yang digunakan:** ...